In [2]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

for name in ["engineered_features_raw", "features_with_causal_states", "household_split"]:
    df = pd.read_csv(ROOT / "data" / "processed" / f"{name}.csv")
    print(name, df.shape)
    print(list(df.columns))
    print(df.head(3).to_string())
    print()

engineered_features_raw (3600, 43)
['household_id', 'month', 'airtime_data', 'entertainment', 'food_groceries', 'housing', 'income_primary', 'income_secondary', 'informal_cash_income', 'loan_repayment', 'mobile_money_transfers', 'savings', 'school_fees', 'transport', 'utilities', 'total_expenditure', 'total_income', 'income_to_expenditure_ratio', 'zero_income_flag', 'housing_share', 'food_groceries_share', 'transport_share', 'utilities_share', 'airtime_data_share', 'mobile_money_transfers_share', 'loan_repayment_share', 'school_fees_share', 'entertainment_share', 'savings_share', 'savings_rate', 'monthly_debt_repayment', 'debt_service_coverage_ratio', 'overspend_flag', 'ratio_roll3', 'overspend_roll3', 'exp_roll3', 'exp_roll6', 'spending_regularity_score', 'household_size', 'num_dependants', 'income_tier', 'dependant_cost_factor', 'true_state']
  household_id  month  airtime_data  entertainment  food_groceries  housing  income_primary  income_secondary  informal_cash_income  loan_repay

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

df = pd.read_csv(ROOT / "data" / "processed" / "engineered_features_raw.csv")
split = pd.read_csv(ROOT / "data" / "processed" / "household_split.csv")
df = df.merge(split, on="household_id")
print(split["split"].value_counts())

CATS = ["airtime_data", "entertainment", "food_groceries", "housing", "loan_repayment",
        "mobile_money_transfers", "savings", "school_fees", "transport", "utilities"]

held = df[df["split"] != "train"]
print("held-out households:", held["household_id"].nunique())

rows = []
for hid, g in held.groupby("household_id"):
    g = g.sort_values("month").set_index("month")
    last = g.loc[9, CATS]
    mean3 = g.loc[7:9, CATS].mean()
    for m in (10, 11, 12):
        for c in CATS:
            rows.append((hid, m, c, g.loc[m, c], last[c], mean3[c]))

res = pd.DataFrame(rows, columns=["household_id", "month", "category", "actual", "last_month", "mean_3m"])

for name in ["last_month", "mean_3m"]:
    res[f"ae_{name}"] = (res["actual"] - res[name]).abs()
    res[f"se_{name}"] = (res["actual"] - res[name]) ** 2

summary = res.groupby("category").agg(
    MAE_last=("ae_last_month", "mean"),
    MAE_mean3=("ae_mean_3m", "mean"),
    RMSE_last=("se_last_month", lambda s: np.sqrt(s.mean())),
    RMSE_mean3=("se_mean_3m", lambda s: np.sqrt(s.mean())),
).round(0)
print(summary)

tot = res.groupby(["household_id", "month"])[["actual", "last_month", "mean_3m"]].sum()
print("Total spend MAE, last month:", round((tot.actual - tot.last_month).abs().mean()))
print("Total spend MAE, 3-month mean:", round((tot.actual - tot.mean_3m).abs().mean()))
print("Average actual total spend:", round(tot.actual.mean()))

split
train    240
test      60
Name: count, dtype: int64
held-out households: 60
                        MAE_last  MAE_mean3  RMSE_last  RMSE_mean3
category                                                          
airtime_data              1159.0     1141.0     3060.0      3150.0
entertainment             1198.0      961.0     3703.0      2807.0
food_groceries            6842.0     6707.0    19928.0     19550.0
housing                      0.0        0.0        0.0         0.0
loan_repayment               0.0        0.0        0.0         0.0
mobile_money_transfers    2274.0     1989.0     5109.0      3958.0
savings                    646.0      795.0     1456.0      1859.0
school_fees               7395.0     2465.0    12034.0      4011.0
transport                 2414.0     2058.0     8140.0      6406.0
utilities                  226.0      185.0      323.0       310.0
Total spend MAE, last month: 14967
Total spend MAE, 3-month mean: 11717
Average actual total spend: 45384


In [4]:
print((df[CATS] == 0).mean().round(2))
print(df.groupby("income_tier")[CATS].mean().round(0).T)


airtime_data              0.00
entertainment             0.00
food_groceries            0.00
housing                   0.52
loan_repayment            0.72
mobile_money_transfers    0.00
savings                   0.00
school_fees               0.45
transport                 0.00
utilities                 0.00
dtype: float64
income_tier             formal_salaried  informal_irregular
airtime_data                     4117.0              1661.0
entertainment                    2843.0              1129.0
food_groceries                  26900.0             13004.0
housing                         10048.0              2261.0
loan_repayment                   5767.0              1370.0
mobile_money_transfers           7831.0              3182.0
savings                          4459.0              1049.0
school_fees                      2984.0              2646.0
transport                        6951.0              3496.0
utilities                        2906.0              1787.0


In [5]:
state_check = df.groupby("true_state").agg(
    households=("household_id", "nunique"),
    months=("month", "size"),
    mean_ratio=("income_to_expenditure_ratio", "mean"),
    median_ratio=("income_to_expenditure_ratio", "median"),
    share_overspend=("overspend_flag", "mean"),
    mean_savings_rate=("savings_rate", "mean"),
).round(2)
print(state_check)
print(pd.crosstab(df["income_tier"], df["true_state"], normalize="index").round(2))

            households  months  mean_ratio  median_ratio  share_overspend  \
true_state                                                                  
Critical           255    1109        0.64          0.73             0.98   
Stable             261    1189        1.11          1.19             0.24   
Strained           286    1302        0.82          0.93             0.69   

            mean_savings_rate  
true_state                     
Critical                 0.02  
Stable                   0.09  
Strained                 0.05  
true_state          Critical  Stable  Strained
income_tier                                   
formal_salaried         0.20    0.53      0.27
informal_irregular      0.36    0.24      0.40


In [6]:
cm_df = pd.read_csv(ROOT / "data" / "processed" / "features_with_causal_states.csv")
t = cm_df[cm_df["split"] == "test"]
order = ["Stable", "Strained", "Critical"]

print(pd.crosstab(t["true_state"], t["hmm_state_causal"]).reindex(index=order, columns=order, fill_value=0))
print(pd.crosstab(t["true_state"], t["hmm_state_causal"], normalize="index").reindex(index=order, columns=order, fill_value=0).round(2))

rank = {"Stable": 0, "Strained": 1, "Critical": 2}
gap = (t["true_state"].map(rank) - t["hmm_state_causal"].map(rank)).abs()
print("exact:", round((gap == 0).mean(), 3),
      "| within one state:", round((gap <= 1).mean(), 3),
      "| Stable<->Critical mixups:", round((gap == 2).mean(), 3))
print("true mix:", t["true_state"].value_counts(normalize=True).round(2).to_dict())

hmm_state_causal  Stable  Strained  Critical
true_state                                  
Stable                83        88        44
Strained              27       120       125
Critical               2        52       179
hmm_state_causal  Stable  Strained  Critical
true_state                                  
Stable              0.39      0.41      0.20
Strained            0.10      0.44      0.46
Critical            0.01      0.22      0.77
exact: 0.531 | within one state: 0.936 | Stable<->Critical mixups: 0.064
true mix: {'Strained': 0.38, 'Critical': 0.32, 'Stable': 0.3}
